# H04 — At home: LIGO data from GWOSC

> **References**
>
> (LIGO Scientific, Virgo and KAGRA Collaborations 2023), (Welch 1967)

## What you do here

In class we measured the noise of Virgo from a file that was already on disk. Here you take the data yourself, from LIGO.

**Part 1, written out for you.** We download one minute of LIGO strain from GWOSC and save it to a file. You run it and read it.

**Then two tasks.**
1. Plot the data in the time domain.
2. Compute and plot its amplitude spectral density (ASD), with Welch's method as in the class demonstration.

**How you work in each task.**
1. Read what we ask.
2. Write your code in the empty cell.
3. Look at the figure and answer the question.

> **The symbols**
>
> - **$h$** — the strain, a number without a unit
> - **$f_{s}$** — the sampling frequency, 4096 Hz
> - **$S(f)$** — the one-sided power spectral density, in Hz$^{-1}$
> - **$\sqrt{S}$** — the amplitude spectral density (ASD), in Hz$^{-1/2}$
> - **$T_{\mathrm{seg}}$** — the length of one segment of Welch's method, in s

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import sys, os
import warnings
# gwpy loads a library that prints a long notice in a notebook; we silence it
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')
from gwpy.timeseries import TimeSeries

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)

from gwcourse_palette import P, apply_style
apply_style()

## Part 1 · Download LIGO data from GWOSC

**GWOSC.** The Gravitational Wave Open Science Center, <https://gwosc.org>, publishes the strain of LIGO, Virgo and KAGRA for everybody to use (CC BY 4.0). The data of the third observing run are described in (LIGO Scientific, Virgo and KAGRA Collaborations 2023).

**How a stretch is chosen.** Three things say which data we want.
- The **detector**: `'H1'` is LIGO Hanford, `'L1'` LIGO Livingston, `'V1'` Virgo.
- The **GPS start and end** of the stretch, in seconds. GPS time counts the seconds from 6 January 1980.
- The **sample rate**: 4096 Hz, as in class.

Our stretch is 64 s of LIGO Hanford from the third observing run, in November 2019. It is ordinary data: no signal is known in it.

**The download.** `TimeSeries.fetch_open_data` from the package `gwpy` asks GWOSC for the stretch and returns it as a `TimeSeries`. A `TimeSeries` holds the strain samples (`.value`), the sample rate (`.sample_rate`), the GPS time of the first sample (`.t0`) and the duration (`.duration`).

**Once only.** We save the stretch to a file and read the file the next time. The download takes about a minute.

> **Careful — This cell needs the network**
>
> The first time it runs it downloads the data. When the file is already there it reads the file and needs no network.

In [ ]:
# the stretch: the detector, the GPS start and end, the sample rate
detector = 'H1'
gps_start = 1257000600          # 6 November 2019, third observing run
gps_end = gps_start + 64        # 64 s of data
sample_rate = 4096              # samples per second (Hz)

# the file where the stretch is kept, next to this notebook
data_file = f'{detector}_{gps_start}_{gps_end - gps_start}s.hdf5'

if os.path.exists(data_file):
    # the stretch is already on disk: read it, no network needed
    data = TimeSeries.read(data_file)
    print(f'read {data_file}')
else:
    # ask GWOSC for the stretch, then save it for the next time
    data = TimeSeries.fetch_open_data(detector, gps_start, gps_end, sample_rate=sample_rate)
    data.write(data_file)
    print(f'downloaded from GWOSC and saved to {data_file}')

# what the TimeSeries holds
strain = data.value                 # the strain samples h[n], a numpy array
fs = data.sample_rate.value         # the sample rate (Hz)
t0 = data.t0.value                  # the GPS time of the first sample (s)
duration = data.duration.value      # the length of the stretch (s)
print(f'{detector}: {strain.size} samples at {fs:.0f} Hz, GPS {t0:.0f}, {duration:.0f} s')
print(f'rms of the strain {strain.std():.2e}')

From here on we work with three names: `strain`, the array of samples; `fs`, the sample rate; `t0`, the start time.

## Task 1 · The data in the time domain

**What we ask.** Plot the strain against time, in seconds from the start of the stretch. Then plot one second of it, to see the samples closely.

**What to think about.** In class the Virgo record was a slow wave with a fast ripple on top. The strain is about $10^{-19}$ or smaller: the motion of a mirror is tiny.

**Your code.**
- Make the time axis: `t = np.arange(strain.size) / fs`.
- Plot `strain` against `t` for the whole stretch, with labels on the axes.
- Plot the first second only.

**The question.** What do you see in time: a white record or a record with structure? Which frequencies seem to carry most of the power?

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Task 2 · The amplitude spectral density

**What we ask.** Estimate the ASD of the stretch with Welch's method and plot it on log axes.

**What to remember from class.**
- Welch's method cuts the record into segments of $T_{\mathrm{seg}}$ seconds, multiplies each by a window and averages the squared Fourier transforms.
- The frequency resolution is $1/T_{\mathrm{seg}}$. Longer segments give a finer resolution and a noisier curve.
- `signal.welch` returns the frequencies and $S(f)$. The ASD is the square root.

**Your code.**
- Call `signal.welch(strain, fs=fs, nperseg=int(4 * fs), window='hann')`: segments of 4 s.
- Take the square root and plot the ASD on log axes, from 10 Hz to `fs / 2`.

**The question.** Where is the detector quietest? Where are the narrow lines and what could they be? LIGO is in the United States, where the mains is at 60 Hz.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## What to hand in

1. **Task 1.** The plot in time, with one or two sentences: what the record looks like and what carries its power.
2. **Task 2.** The plot of the ASD, with one or two sentences: where the detector is quietest and what the strongest lines could be.

## Software

`gwpy` downloads the data and saves it; `numpy` and `scipy.signal.welch` give the spectrum, as in the class demonstration. The data come from GWOSC (CC BY 4.0) and the third observing run is described in (LIGO Scientific, Virgo and KAGRA Collaborations 2023). Welch's method is (Welch 1967).

**References**

- LIGO Scientific, Virgo and KAGRA Collaborations (2023). *Open Data from the Third Observing Run of LIGO, Virgo, KAGRA, and GEO*.
- Welch (1967). *The use of fast Fourier transform for the estimation of power spectra: A method based on time averaging over short, modified periodograms*.